# Lab 06 — Preparing a CORD receipt annotation pilot

Choose the OCR line containing the final total amount, or **No total found**. This pilot uses training receipts from the prepared Project 04 dataset. Run from `labs/Lab06`.

## 1. Load and inspect receipts

The manifest identifies images and splits; `documents.jsonl` contains stable line IDs, OCR text and private reference annotations. The flat OCR table is used by the Project 04 baseline.

In [ ]:
from pathlib import Path
from urllib.parse import quote
from io import BytesIO
import json
import os
import shutil
import sys

import pandas as pd
import requests
from PIL import Image, UnidentifiedImageError
from IPython.display import display

LAB_DIR = Path.cwd().resolve()
REPO_ROOT = LAB_DIR.parents[1]
sys.path.insert(0, str(LAB_DIR))
from utils.assignment import prepare_image_for_assignment

DATA_DIR = REPO_ROOT / "project_baselines/project_04/data/prepared/cord_v2"
DOCUMENT_ROOT = (REPO_ROOT / "data").resolve()
manifest = pd.read_csv(DATA_DIR / "manifest.csv")
lines = pd.read_csv(DATA_DIR / "ocr_lines.csv").fillna({"text": ""})
with (DATA_DIR / "documents.jsonl").open(encoding="utf-8") as f:
    documents = {doc["document_id"]: doc for doc in map(json.loads, f)}
train = manifest[manifest["source_split"] == "train"].copy()
print("Training receipts:", len(train), "| OCR rows:", len(lines))
display(train.head())

In [ ]:
example = documents[train.iloc[0]["document_id"]]
with Image.open(DATA_DIR / example["image"]) as image:
    display(image.copy())
display(pd.DataFrame(example["lines"])[["line_id", "text", "course_role"]])

## 2. Select training receipts

Change the sample size or seed to explore another pilot. Sampling does not use the reference labels, so receipts without an annotated total remain eligible.

In [ ]:
SAMPLE_SIZE = 18
RANDOM_SEED = 42
NUM_WORKERS = 3
NUM_RATINGS_PER_EXAMPLE = 2

pilot = train.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED).copy().reset_index(drop=True)
display(pilot[["document_id", "image", "source_split"]])

## 3. Copy images and build local-file URLs

The paths beneath the prepared dataset are preserved under `data/labelstudio/cord_v2/`. Configure that directory as local storage in Label Studio without syncing it as separate tasks.

In [ ]:
DEST_DIR = DOCUMENT_ROOT / "labelstudio/cord_v2"

def copy_image(relative_path):
    relative = Path(relative_path)
    source = (DATA_DIR / relative).resolve()
    destination = (DEST_DIR / relative).resolve()
    if not source.is_relative_to(DATA_DIR.resolve()) or not destination.is_relative_to(DEST_DIR.resolve()):
        raise ValueError(f"Image path outside dataset: {relative}")
    if not source.is_file():
        raise FileNotFoundError(source)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists() or source.read_bytes() != destination.read_bytes():
        shutil.copy2(source, destination)
    return "/data/local-files/?d=" + quote(destination.relative_to(DOCUMENT_ROOT).as_posix(), safe="/")

pilot["image_url"] = pilot["image"].map(copy_image)
print("Copied or verified", len(pilot), "receipt images under", DEST_DIR)
display(pilot[["document_id", "image_url"]].head())

## 4. Assign workers and build tasks

As in the previous examples, we assign each receipt to a number of annotators.

However, this task is different from text or image classification. Instead of choosing from a fixed set of labels, annotators must **identify which OCR line contains the receipt's total amount**.

We use the existing OCR data to generate a list of choices for each receipt. Each choice includes:

- **Line ID:** A unique identifier for the OCR line, which allows us to match the annotation with the original data.
- **OCR text:** The recognized text shown to the annotator.

We also include a **No total found** option for cases where none of the available OCR lines contains the total. This does not necessarily mean that the total is missing from the receipt itself; the OCR system may have failed to recognize it.

In [ ]:
selected, assigned_tasks, assignments = prepare_image_for_assignment(
    pilot,
    id_col="document_id",
    image_col="image_url",
    sample_size=len(pilot),
    num_workers=NUM_WORKERS,
    num_ratings_per_example=NUM_RATINGS_PER_EXAMPLE,
    seed=RANDOM_SEED,
)

# Generate the available answers for each receipt

NO_TOTAL = "No total found"

def make_options(document_id):
    receipt_lines = documents[document_id]["lines"]

    ids = [line["line_id"] for line in receipt_lines]
    if len(ids) != len(set(ids)):
        raise ValueError(f"Duplicate line IDs in {document_id}")

    return [
        {
            "value": f"{line['line_id']} | {line['text']}",
            "alias": line["line_id"]
        }
        for line in receipt_lines
    ] + [
        {"value": NO_TOTAL, "alias": "no_total"}
    ]

# build the label studio tasks

task_records = [
    {
        "data": {
            "example_id": row.example_id,
            "image": row.image,
            "assigned_workers": row.assigned_workers,
            "options": make_options(row.example_id)
        }
    }
    for row in assigned_tasks.itertuples(index=False)
]

### Prepare OCR bounding boxes

So far, we have prepared a task where annotators select the correct OCR line from a list. However, our dataset also contains the **location of each OCR line** on the receipt.

We can use this information to create an alternative annotation interface. Instead of reading a list of OCR lines, annotators can select the corresponding bounding box directly on the image.

To do this, we convert the OCR coordinates into the format expected by Label Studio and include them as **preannotations (predictions)**.

We then create a second version of our tasks, keeping the same images and worker assignments but replacing the list of options with preloaded bounding boxes.

**Note:** All OCR boxes are included, without revealing which one contains the total amount. The annotator must still identify the correct box.

In [ ]:
def make_ocr_boxes(document_id):
    doc = documents[document_id]
    width = doc["image_width"]
    height = doc["image_height"]

    boxes = []

    for line in doc["lines"]:
        x1, y1, x2, y2 = line["bbox"]

        boxes.append({
            "id": line["line_id"],
            "from_name": "ocr_box",
            "to_name": "receipt",
            "type": "rectangle",
            "original_width": width,
            "original_height": height,
            "image_rotation": 0,
            "value": {
                "x": 100 * x1 / width,
                "y": 100 * y1 / height,
                "width": 100 * (x2 - x1) / width,
                "height": 100 * (y2 - y1) / height,
                "rotation": 0
            }
        })

    return boxes

box_tasks = []

for task in task_records:
    data = task["data"]
    document_id = data["example_id"]

    box_tasks.append({
        "data": data,
        "predictions": [{
            "model_version": "cord_ocr_boxes",
            "result": make_ocr_boxes(document_id)
        }]
    })

print("Tasks:", len(box_tasks))
print("OCR boxes in first task:", len(box_tasks[0]["predictions"][0]["result"]))


In [ ]:
display(pd.DataFrame([task["data"] for task in task_records]).head())
display(assignments["worker_id"].value_counts().rename("assignments"))

## 5. Export tasks and private references

Import only the JSON task file. The reference stores all source `total_amount` line IDs (an empty list means none was annotated), plus the source parsed values for later inspection. It is not shown to annotators.

In [ ]:
# Prepare reference data
reference_rows = []

for doc_id in selected["document_id"]:
    doc = documents[doc_id]

    total_lines = [
        line["line_id"]
        for line in doc["lines"]
        if line["course_role"] == "total_amount"
    ]

    reference_rows.append({
        "example_id": doc_id,
        "reference_total_line_ids": json.dumps(total_lines),
        "gt_parse": json.dumps(doc.get("gt_parse"), ensure_ascii=False)
    })

reference = pd.DataFrame(reference_rows)
reference.head()

In [ ]:
from pathlib import Path

EXPORT_DIR = Path("export")
EXPORT_DIR.mkdir(exist_ok=True)

# Label Studio tasks
with open(EXPORT_DIR / "cord_tasks.json", "w", encoding="utf-8") as f:
    json.dump(box_tasks, f, ensure_ascii=False, indent=2)

# Private reference labels and worker assignments
reference.to_csv(EXPORT_DIR / "cord_reference.csv", index=False)
assignments.to_csv(EXPORT_DIR / "cord_assignments.csv", index=False)

## 6. Configure Label Studio

Create a new Label Studio project to annotate the receipts.

1. **Configure the labeling interface:** Use `label_studio_setup/cord_receipt.xml`. The interface displays the receipt image and its OCR lines as selectable options.
2. **Connect the images:** Go to **Project Settings → Cloud Storage → Add Source Storage**. Select **Local Files** and set the path to `labelstudio/cord_v2/images/train` (relative to the configured local-files document root).
3. **Import the tasks:** Import `export/cord_tasks.json` into the project. Do not synchronize the storage folder as separate tasks, as this would create duplicates.
4. **Configure annotators:** Use the worker assignments to create filtered tabs in the Data Manager. Set the required number of annotations per task using `scripts/configure_lstudio.py` from the repository root.

Keep `cord_reference.csv` private so annotators cannot see the original labels.